# Prepare the Spatial Metabolomics data (using zonist)

Our "spatially aware" pathways analysis, for a given **spatial metabolomics dataset**, requires:
1. Formatting the data to be appropriate for pathway scores computation
2. To obtain the matrix of pathways (or metabolite sets) activity scores across all pixels on the correctly formatted data, and
3. To perform the downstream analysis of such matrix of pathways scores

However, preliminary steps needed for obtaining 1. are multiple: (i) combining disctint annotations, (ii) translation of the annotation ids to KEGG ids, (iii) gmt file preparation, (iv) verify and clean the quantification matrix.

This notebook shows step-by-step the use of zonit to **go faster !** through the preliminary data preparation:

1. Formatting the data to be appropriate for pathway scores computation
   * **i)** loading and formatting of the annotations table(s) corresponding to the metabolome to be analyzed
   * **ii)** loading and formatting the metabolites reference mappings to **KEGG** identifiers: the object is then used to perform the **conversion** of the "no-kegg" identifiers, present in the metabolome annotations (i), to KEGG ones. 
   * **iii)** generation of a metabolome-tailored **.gmt** file, that specifies **mz** features across pathways
   * **iv)** verification of mz x pixels quantification matrix and save a clean version ready for GSVA


In [1]:
import os
import numpy as np
import pandas as pd
import csv as csv

In [2]:
import zonist.data as zd

INFO:zonist.data:
Time: 20241028-12.11.23


## i ) Load annotations

_Note_: : allows files generated by Metaspace, and also manually curated and/or prepared files having the compulsory columns (mz, formula, moleculeIds, moleculeNames, database)


In [3]:
mydata_dir = "/home/johanna/spatial_thesis/spatial_metabolomics/data"
#"/mnt/cbib/thesis_gbm/spatial_thesis/spatial_metabolomics/data"
anno_dict = {'HMDB': "brain-ellis_hmdb_annotations.csv",
             'LipidMaps': "brain-ellis_lipidmaps_annotations.csv",
            'custom_kegg': "brain-ellis_kegg_annot-manually.csv"}  

In [4]:
me_annots = zd.MetabolomeAnnotations(name="demo", 
                            data_dir=mydata_dir, 
                            subfolder="brain-ellis",
                            annotation_files=anno_dict)

In [5]:
me_annots

MetabolomeAnnotations(name='demo', data_dir='/home/johanna/spatial_thesis/spatial_metabolomics/data', subfolder='brain-ellis', annotation_files={'HMDB': 'brain-ellis_hmdb_annotations.csv', 'LipidMaps': 'brain-ellis_lipidmaps_annotations.csv', 'custom_kegg': 'brain-ellis_kegg_annot-manually.csv'}, delimiter=',', col_molecule_ids='moleculeIds', col_mz='mz', col_formula='formula', col_molecule_names='moleculeNames')

In [6]:
me_annots.load_annotation()

INFO:zonist.data:Loading annotations
INFO:zonist.data:  loading LipidMaps
INFO:zonist.data:  loading custom_kegg
INFO:zonist.data:  loading HMDB


In [7]:
me_annots.annotationtbl.head(3)

,mz,formula,moleculeIds,moleculeNames,database
0,369.351578,C27H44,LMST03020618,3-Deoxyvitamin D3,LipidMaps
1,551.503387,C35H66O4,LMGL02040001,DG(P-14:0/18:1(9Z)),LipidMaps
2,720.553782,C39H78NO8P,LMGP02011252,PE(13:0/21:0),LipidMaps


In [8]:
me_annots.annotationtbl['database'].unique()

array(['LipidMaps', 'KEGG', 'HMDB'], dtype=object)

In [9]:
me_annots.annotationtbl.shape

(5682, 5)

There are many K of rows because for each mz there can be tenths of metabolites as candidates.

In fact, the mz are much more less

In [10]:
len(me_annots.annotationtbl['mz'].unique())

311

_question_: the number of annotations that are not KEGG, and those being KEGG

In [11]:
nb_no_kegg = len(me_annots.annotationtbl.loc[
    me_annots.annotationtbl["database"] !="KEGG", 'moleculeIds'].unique())
nb_kegg = len(me_annots.annotationtbl.loc[
    me_annots.annotationtbl["database"] =="KEGG", 'moleculeIds'].unique())
print(f'No-KEGG :{nb_no_kegg};  KEGG :{nb_kegg}')

No-KEGG :5679;  KEGG :3


keep this "untranslated" annotations in a separate variable to make some comparisons later

In [12]:
untranslated_annots = me_annots.annotationtbl.copy()

## ii) Load reference mappings to KEGG identifiers, and perform no-kegg identifiers translation


### *First*: Load the reference mappings of metabolites identifiers from "no-KEGG" to KEGG

In [13]:
ref_files_dict = {'HMDB': "HMDB-to-KEGG_05-07-24.tsv",
              'LipidMaps': "LipidMaps-to-KEGG_10-18-24.tsv"}

In [14]:
me_refmap = zd.MetaboliteReferenceMappings(
    data_dir=mydata_dir,
    subfolder="metabolites-mappings",
    mapping_files=ref_files_dict
)

In [15]:
me_refmap.load_mappings()

INFO:zonist.data:Loading reference files for mapping compounds identifiers
INFO:zonist.data:  loading LipidMaps
INFO:zonist.data: your loaded dataframe :
       LipidMaps    KEGG
0  LMFA08030003  C11839
1  LMFA08030004  C11841
INFO:zonist.data:  loading HMDB
INFO:zonist.data: your loaded dataframe :
           HMDB    KEGG
0  HMDB0000001  C01152
1  HMDB0000002  C00986


### *Second*: convert the metabolome annotations using the loaded reference mappings

To recall, the `.annotationtbl` attribute of the annotations object contains the HMDB and LipidMaps identifiers (which must be convertd to KEGG ones):


In [16]:
me_annots.annotationtbl.head(2)

,mz,formula,moleculeIds,moleculeNames,database
0,369.351578,C27H44,LMST03020618,3-Deoxyvitamin D3,LipidMaps
1,551.503387,C35H66O4,LMGL02040001,DG(P-14:0/18:1(9Z)),LipidMaps


Do the translation effortless with this simple line ! :


In [17]:
me_annots.convert_ids_to_kegg(me_refmap.table)

INFO:zonist.data:Translating ids in the metabolome annotation table, to KEGG


Note that the new column **KEGG** appears in the table:

In [18]:
me_annots.annotationtbl.sample(2)

,mz,formula,moleculeIds,moleculeNames,KEGG,database
652,784.666095,C46H89NO8,HMDB0010711,Galactosylceramide (d18:1/22:0),C02686,HMDB
2358,301.216207,C20H28O2,HMDB0006219,13-cis-Retinoic acid,C07058,HMDB


In [19]:
# the above added the column KEGG to the .annotationtbl, automatically
me_annots.annotationtbl.shape

(236, 6)

-----------------------
_Questions_

_question_ : how many initial molecule identifiers were there, before conversion?

In [20]:
len(untranslated_annots['moleculeIds'].unique())

5682

_question_ : how many of the initial molecule identifiers could NOT be convertd to KEGG ?

In [21]:
unconvertd_set = set(
    untranslated_annots['moleculeIds'].unique()) - set(
    me_annots.annotationtbl['moleculeIds'].unique())
len(unconvertd_set)

5446

_question_ : how many of the initial molecule identifiers could be convertd to KEGG ?

In [22]:
len(me_annots.annotationtbl['moleculeIds'].unique())

236

_question_ : how many unique mz are retained now (their mol identifiers convertd to KEGG) ? What does this amount represent in percentage ?

In [23]:
a_percnt = np.around(len(me_annots.annotationtbl['mz'].unique()) / 
             len(untranslated_annots["mz"].unique()) *100, 1)
print(f"{len(me_annots.annotationtbl['mz'].unique())} mz features :"
      f" the {a_percnt} % of the initial mz features" )

122 mz features : the 39.2 % of the initial mz features


## iii) GMT 

In [24]:
me_gmt = zd.MetabolicGmt(
    data_dir=mydata_dir,
    subfolder="metabolites-mappings",
    official_file="KEGG_hsa_pathways_compounds_R110.gmt",
    custom_file="user-defined-pathways-or-sets.gmt")

In [25]:
me_gmt.load()

INFO:zonist.data: The first line of your custom gmt file:
  abbreviation: 'usr-lac';   name : 'Lactate metabolism (Lac,Pyr,acCoA,LacCoA)';  metabolites  (first three) : '['C00186', 'C00022', 'C00024']'.
INFO:zonist.data:success, attributes : gmt_piled, pathways_ids


In [26]:
me_gmt.gmt_piled.sample(1)

,pathway_id,KEGG
3814,hsa00983,C16632


In [27]:
me_gmt.pathways_ids.sample(1)

,pathway_id,pathway_name
222,hsa05218,Melanoma


In [28]:
me_gmt.join_mz(me_annots.annotationtbl[["mz", "KEGG"]])

In [29]:
me_gmt.gmt_piled.head(1)

,pathway_id,KEGG,mz
0,hsa00100,C05437,385.346492


In [30]:
me_gmt.gmt_piled.shape

(1227, 3)

--------------------------------
_Questions_

_question_ : how many of the KEGG convertd _mz_ are NOT associated to any pathway ? :

In [31]:
len(set(me_annots.annotationtbl['mz'].unique()) - set(
    me_gmt.gmt_piled['mz'].unique()))

22

_question_ : how many of the KEGG convertd _mz_ are associated to at least one pathway ? :

In [32]:
print(len(me_gmt.gmt_piled['mz'].unique()))

100


In [33]:
me_gmt.write_to_json(
    dataset_subfolder="brain-ellis", mz_or_kegg_choice="mz")

INFO:zonist.data:saved json file to : /home/johanna/spatial_thesis/spatial_metabolomics/data/brain-ellis/gmt_mz.json


## iv) Quantifications matrix

The quantification matrix must have unique $mz$ features that perfectly match with those in the annotations. Two alternative ways of obtaining this matrix:

1. from the imzML+ibd files: this requires careful formatting as the $mz$ features in this file are never identical to those in the annotations (because the annotations keep the theoretical 'm/z' of the databases used by Metaspace).
OR,
2. by downloading the matrix from Metaspace and formatting it.

The matrix treated here was obtained as specified in numeral 2. above.

_Note_:
The matrix in our example from Metaspace has doublets of $mz$ features, because several databases (LipidMaps and HMDB) yielded hits for a same row (thus duplicating the row). Moreover, there are supplementary columns that GSVA will not accept: _formula_, _moleculeNames_, _adduct_, etc.  To run GSVA it is compulsory to have unique $mz \cdot coords$. This means, in rows, the $mz$ features, and in columns the 'x,y' coordinates ($coords$).

In [34]:
pre_mx_df = pd.read_csv(
    os.path.join(mydata_dir, "brain-ellis", 
                 "brain-ellis-mx-subset.csv"), 
    sep=",", comment="#", header=0)

In [35]:
print(f"mz with doublets: {len(pre_mx_df['mz'])}")
print(f"mz as unique features: {len(pre_mx_df['mz'].unique())}")

mz with doublets: 407
mz as unique features: 310


In [36]:
pre_mx_df.columns[:7]

Index(['mol_formula', 'adduct', 'mz', 'moleculeNames', 'moleculeIds', 'x0_y0',
       'x0_y1'],
      dtype='object')

In [37]:
cols_keep = ['mz']
cols_keep += pre_mx_df.columns[5:].tolist()

In [38]:
print("current shape : ", pre_mx_df.shape)

current shape :  (407, 171155)


In [39]:
print(f"expected matrix shape : {len(pre_mx_df['mz'].unique())} x {len(cols_keep)}")

expected matrix shape : 310 x 171151


In [40]:
mx_df = pre_mx_df[cols_keep]
del pre_mx_df

In [41]:
mx_df.iloc[:3,:3]

,mz,x0_y0,x0_y1
0,233.226377,0,0
1,237.221292,0,0
2,243.065185,0,0


In [42]:
mx_df = mx_df.sort_values(by='mz', ascending=True)

In [43]:
mx_df = mx_df.drop_duplicates()

In [44]:
mx_df.shape

(310, 171151)

<div class="alert alert-block alert-danger">
<b>ALERT! Too many columns:</b> Files with ~200K of columns are slow to open (or write) in Python, and tremendously slow to open (or write) in R (~30 minutes !!). Moreover, in spatial metabolomics the mz features will be always less than 10K. To save time when reading/writing files with such a huge amount of columns, the transposed version is saved as .tsv file.
</div>

In [45]:
mx_df.T.to_csv(os.path.join(mydata_dir, "brain-ellis", 
                 "brain-ellis-mx-clean-transposed.tsv"), 
               quoting=csv.QUOTE_NONNUMERIC, 
               sep="\t", index=True, header=False)

The GSVA step is treated in next notebook

In the terminal, from your home or working directory, you launch the R script:
```
Rscript zonist/src/r/$THE_SCRIPT ~/spatial_thesis/spatial_metabolomics/data/brain-ellis brain-ellis-mx-clean-transposed.tsv gmt_mz.json
```
it requires 13 hours of computing time, despite parallelization.